In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

# 1. Загрузка данных и знакомство с ними
data = pd.read_csv("adult.csv")

print("Размер таблицы:", data.shape)
display(data.head())
print("\nТипы данных и количество непустых значений:")
data.info()

# Очистка скрытых пропусков
data = data.replace("?", np.nan)

target = "income"
X = data.drop(columns=[target])
y = data[target]

print("Распределение целевой переменной:")
print(y.value_counts())

# 2. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

# 3. Предварительная обработка признаков
numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(exclude=["number"]).columns

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 4. Модель логистической регрессии
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

logistic_model.fit(X_train, y_train)

logistic_score = logistic_model.score(X_test, y_test)
print(
    f"Качество логистической регрессии на тестовой выборке: {logistic_score:.4f}"
)

# 5. Модель SVM
svm_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LinearSVC(max_iter=5000, random_state=42)),
    ]
)

svm_model.fit(X_train, y_train)

svm_score = svm_model.score(X_test, y_test)
print(f"Качество SVM на тестовой выборке: {svm_score:.4f}")

# 6. Сравнение результатов
results = pd.DataFrame(
    {
        "Модель": ["Логистическая регрессия", "SVM (LinearSVC)"],
        "Качество на тестовой выборке (.score)": [logistic_score, svm_score],
    }
)

display(
    results.sort_values(
        by="Качество на тестовой выборке (.score)", ascending=False
    ).reset_index(drop=True)
)

Размер таблицы: (48842, 15)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K



Типы данных и количество непустых значений:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB
Распределение целевой переменно

,Модель,Качество на тестовой выборке (.score)
0,SVM (LinearSVC),0.852902
1,Логистическая регрессия,0.852390
